In [0]:
%sql
CREATE OR REPLACE TABLE hospital.gold.visit_readmission
COMMENT 'Une ligne par visite, avec indicateur de réadmission à 30 jours (attribuée au séjour d origine)'
AS
WITH ordered AS (
  SELECT v.*,
         LEAD(v.admission_date) OVER (PARTITION BY v.patient_id ORDER BY v.admission_date) AS next_admission_date
  FROM hospital.silver.fact_visit v
)
SELECT
  o.visit_id, o.patient_id, o.hospital_id, o.diagnosis_code,
  h.hospital_name, h.city AS hospital_city, h.bed_count,
  d.diagnosis_desc,
  o.admission_date, o.discharge_date, o.length_of_stay, o.cost,
  o.next_admission_date,
  datediff(o.next_admission_date, o.discharge_date) AS days_to_readmission,
  CASE WHEN datediff(o.next_admission_date, o.discharge_date) BETWEEN 0 AND 30
       THEN 1 ELSE 0 END AS is_readmitted_30d,
  current_timestamp() AS _computed_at
FROM ordered o
LEFT JOIN hospital.silver.dim_hospital h
  ON  o.hospital_id = h.hospital_id
  AND CAST(o.admission_date AS TIMESTAMP) >= h.valid_from
  AND CAST(o.admission_date AS TIMESTAMP) <  COALESCE(h.valid_to, TIMESTAMP '9999-12-31')
LEFT JOIN hospital.silver.dim_diagnosis d
  ON o.diagnosis_code = d.diagnosis_code

In [0]:
%sql
CREATE OR REPLACE TABLE hospital.gold.hospital_disease_kpi
COMMENT 'KPI par hôpital et diagnostic : visites, réadmissions 30 j, coûts, durée de séjour'
AS
SELECT
  hospital_id, hospital_name, hospital_city,
  diagnosis_code, diagnosis_desc,
  COUNT(*)                                       AS total_visits,
  SUM(is_readmitted_30d)                         AS total_readmissions_30d,
  ROUND(SUM(is_readmitted_30d) / COUNT(*), 3)    AS readmission_rate,
  SUM(cost)                                      AS total_cost,
  ROUND(AVG(cost), 2)                            AS avg_cost,
  ROUND(AVG(length_of_stay), 1)                  AS avg_length_of_stay,
  current_timestamp()                            AS _computed_at
FROM hospital.gold.visit_readmission
GROUP BY ALL

In [0]:
%sql
SELECT hospital_name, diagnosis_desc, total_visits, total_readmissions_30d, readmission_rate, total_cost
FROM hospital.gold.hospital_disease_kpi
ORDER BY readmission_rate DESC, total_cost DESC

In [0]:
%sql
select * from  hospital.gold.visit_readmission

In [0]:
%sql
SELECT hospital_name, diagnosis_desc, total_visits, total_readmissions_30d, readmission_rate, total_cost
FROM hospital.gold.hospital_disease_kpi
ORDER BY readmission_rate DESC, total_cost DESC